# Lab: Decision Trees, Bagging, Random Forests and Boosting

Based on the Chapter 8 lab (sections 8.3.1-8.3.4). We fit a classification tree to `Carseats` and regression trees
and ensembles to `Boston`, first with `scikit-learn`, then recomputing the key steps by hand.

Cells marked `# TODO` must be completed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import confusion_matrix, accuracy_score, mean_squared_error

## 1. Classification trees

### 1.1 The Carseats dataset

Sales of child car seats in 400 stores. `Sales` is continuous (thousands of units), so we create a binary
response `High` = 1 if `Sales` > 8. Trees in `scikit-learn` need numerical inputs: encode `ShelveLoc` (quality of
the shelving location: Bad, Medium, Good) as 0, 1, 2, and `Urban`, `US` as 0/1.

In [ ]:
carseats = pd.read_csv("datasets/Carseats.csv")
carseats["High"] = (carseats["Sales"] > 8).astype(int)
carseats["ShelveLoc"] = carseats["ShelveLoc"].map({"Bad": 0, "Medium": 1, "Good": 2})
carseats["Urban"] = (carseats["Urban"] == "Yes").astype(int)
carseats["US"] = (carseats["US"] == "Yes").astype(int)
carseats.head()

Use 200 stores for training and 200 for testing. `Sales` must not be a predictor (it defines the response).

In [ ]:
X = carseats.drop(columns=["Sales", "High"])
y = carseats["High"]
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=200, random_state=0)
print("proportion of High in the training set:", y_train.mean())

### 1.2 Fitting a tree

Fit a tree with the Gini index, `max_depth=6` and `min_samples_leaf=4` (as in the reference lab). Report the training
accuracy and draw the first levels of the tree.

In [ ]:
# TODO
tree_gini = ...

In [ ]:
plt.figure(figsize=(16, 7))
plot_tree(tree_gini, feature_names=list(X.columns), max_depth=2, filled=True, fontsize=9)
plt.show()

- Which variable is used at the root? Does it make sense as the most important predictor of sales?
- Read one path from the root to a leaf in words.

Now compute the confusion matrices (rows: true class, columns: predicted class) and the accuracy on the training
and on the test set.

In [ ]:
# TODO

- Compare train and test accuracy. What is happening?

### 1.3 The first split, by hand

The root split is the pair (feature $j$, threshold $s$) that minimizes the weighted Gini index of the two children

$$\frac{n_L}{n}\,G(L) + \frac{n_R}{n}\,G(R), \qquad G = \sum_k \hat p_k (1 - \hat p_k) = 2\hat p(1 - \hat p) \ \text{(two classes)}.$$

Write `gini(y)` and `best_split(X, y)`, which tries every feature and every midpoint between consecutive distinct
values, and returns `(feature name, threshold, weighted Gini)`. Compare with `tree_gini.tree_.feature[0]` and
`tree_gini.tree_.threshold[0]`.

In [ ]:
def gini(y):
    # TODO
    ...


def best_split(X, y):
    # TODO
    ...

### 1.4 Pruning with cross-validation

The depth 6 was chosen arbitrarily. Instead, grow a full tree and prune it: for each $\alpha$, keep the subtree that
minimizes (error) $+\ \alpha\,|T|$. `cost_complexity_pruning_path` returns the values of $\alpha$ where the optimal
subtree changes.

Choose `ccp_alpha` by 5-fold cross-validation on the training set (`GridSearchCV`, accuracy), plot the CV accuracy
against $\alpha$, then report the number of leaves and the test accuracy of the selected tree.

In [ ]:
full_tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
path = full_tree.cost_complexity_pruning_path(X_train, y_train)
print("full tree:", full_tree.get_n_leaves(), "leaves, test accuracy", full_tree.score(X_test, y_test))

# TODO

In [ ]:
# TODO: draw the pruned tree

- How does the pruned tree compare with the full tree and with the depth-6 tree, in size and test accuracy?
- Repeat the CV with a different `random_state` in `KFold`. Is the chosen $\alpha$ stable? What does that say about
  trees?

## 2. Regression trees

### 2.1 The Boston dataset

Median house value `medv` (in thousands of dollars) of 506 Boston suburbs, with 12 predictors (crime rate, average
number of rooms `rm`, percentage of lower status population `lstat`, ...). Use half of the data for training.

In [ ]:
boston = pd.read_csv("datasets/Boston.csv", index_col=0)
X_b = boston.drop(columns="medv")
y_b = boston["medv"]
Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_b, y_b, train_size=0.5, random_state=0)
p = X_b.shape[1]
boston.head()

Fit a regression tree with `max_depth=10`, draw its first levels, and compute the test MSE.

In [ ]:
# TODO
regr_tree = ...
mse_tree = ...

- The square root of the MSE is in the units of `medv`. What is the typical prediction error in dollars?
- Which variables appear at the top of the tree?

Prune the tree with 5-fold CV over `ccp_alpha` (scoring `"neg_mean_squared_error"`) and compare the test MSE.

In [ ]:
# TODO

## 3. Bagging and random forests

### 3.1 Bagging

Bagging is a random forest that uses all $p$ features at every split. Fit `RandomForestRegressor` with
`max_features=p` (100 trees) and compute the test MSE.

In [ ]:
# TODO
bag = ...
mse_bag = ...

### 3.2 Bagging by hand

Now build the bagged ensemble yourself: for $b = 1, \dots, B$ draw a bootstrap sample (n indices with replacement),
fit a full `DecisionTreeRegressor` on it, and average the $B$ predictions on the test set. Plot the test MSE of the
average of the first $b$ trees as a function of $b$.

In [ ]:
B = 100
rng = np.random.default_rng(0)
Xtr, ytr = Xb_train.to_numpy(), yb_train.to_numpy()
# TODO

- How many trees are needed before the curve flattens? Is the gain over a single tree large?
- The two final MSEs are close but not equal. Why?

In [ ]:
# TODO
rf = ...
mse_rf = ...

Try every value of `max_features` from 1 to $p$ and plot both the test MSE and the OOB MSE.

In [ ]:
# TODO

- Does a smaller $m$ help on this dataset? Do the OOB and the test curves agree on the best $m$?
- Why is the OOB error a legitimate estimate of the test error, even though it is computed on the training set?

### 3.4 Variable importance

Plot `feature_importances_` of the random forest (mean decrease in RSS due to splits on each variable, normalized).

In [ ]:
# TODO

- Which two variables dominate? Are they the ones at the top of the single tree?

## 4. Boosting

### 4.1 With scikit-learn

Fit `GradientBoostingRegressor` with 1000 trees, `learning_rate=0.01` and `max_depth=3`. Report the test MSE and
plot the variable importance.

In [ ]:
# TODO
boost = ...
mse_boost = ...

### 4.2 Boosting by hand

Implement the algorithm of the lecture with `DecisionTreeRegressor(max_depth=3)` as base learner:

1. $\hat f(x) = \bar y$ (`scikit-learn` starts at the mean), $r_i = y_i - \bar y$.
2. For $b = 1, \dots, B$: fit a tree to $(x_i, r_i)$, then $\hat f \leftarrow \hat f + \lambda \hat f^b$ and
   $r_i \leftarrow r_i - \lambda \hat f^b(x_i)$.

Keep the test MSE after each step and compare the final value with `mse_boost`.

In [ ]:
lam, B = 0.01, 1000
Xte = Xb_test.to_numpy()
# TODO

### 4.3 The learning rate

Use `staged_predict` to plot the test MSE against the number of trees for `learning_rate` 0.01, 0.1 and 0.5
(1000 trees, `max_depth=3`).

In [ ]:
# TODO

- With a large learning rate, where is the minimum and what happens afterwards?
- How would you choose the number of trees without a test set?

In [ ]:
try:
    import xgboost as xgb

    model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.01, max_depth=6).fit(Xb_train, yb_train)
    print("XGBoost test MSE:", mean_squared_error(yb_test, model.predict(Xb_test)))
except ImportError:
    print("xgboost is not installed")

## 5. Comparison

Collect the test MSE of all the regression models in a table.

In [ ]:
# TODO

- Rank the methods. Is the ranking the same as in the reference lab?
- A test set of 253 houses is small. How could you make the comparison more reliable?